<a href="https://colab.research.google.com/github/WiebkePetersen/GermevalFlausch/blob/main/flausch_task1_train_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# only for Colab
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [2]:
# auf Google Colab muss numpy Version runtergesetzt werden, und anschließend Sitzung neu gestartet werden,
# damit Task 1 läuft. Sonst kommt es im Training zu Fehlerabbrüchen
!pip install numpy==1.26.4


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 79.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


## Flausch Detection: Task 1: binary comment classification -- train BERT model

In [1]:
import pandas as pd
import os
import sys
import numpy as np

# work with the train data only
path = ""
#path = "Input/Data/train/"
#path = "https://raw.githubusercontent.com/WiebkePetersen/GermevalFlausch/tree/main/Input/Data/train/"


data = pd.read_json(path + "train_task1_new.json")
data2 = pd.read_json(path + "train_task2_new.json")


In [2]:
import pandas as pd
from datasets import Dataset, DatasetDict


id2label = {0: 'no', 1: 'yes'}
label2id = {'no': 0, 'yes': 1}
data['labels'] = data['flausch'].map(label2id)

# Convert to Hugging Face Dataset

dataset = Dataset.from_pandas(data)



In [3]:
dataset

Dataset({
    features: ['document', 'comment_id', 'comment', 'flausch', 'translated', 'spans', 'span_pairs', 'types', 'id', 'spelling_corrected', 'sentiment_orig', 'sentiment_spelling_corrected', 'sentiment_translated', 'labels', '__index_level_0__'],
    num_rows: 33351
})

In [4]:
import torch
from transformers import AutoTokenizer

checkpoint = "deepset/gbert-large"
#checkpoint = "bert-base-german-cased"

column = "spelling_corrected" #"comment"

model_name = checkpoint.split("/")[-1] + "_" + column

tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(examples):
    # Der Tokenizer gibt hier Listen von Listen zurück (für den Batch)
    tokenized_inputs = tokenizer(examples[column], truncation=True, max_length=512)
    return tokenized_inputs

# remove columns not needed for tokenization



# Tokenize dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

keep_cols = ['labels', 'input_ids', 'attention_mask']

# Setze das Format auf PyTorch-Tensoren
tokenized_dataset.set_format("torch")
tokenized_dataset = tokenized_dataset.remove_columns([col for col in tokenized_dataset.column_names if col not in keep_cols])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/83.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/240k [00:00<?, ?B/s]

Map:   0%|          | 0/33351 [00:00<?, ? examples/s]

In [5]:
tokenized_dataset

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 33351
})

In [6]:
# data consists of 90% of the data
# 85% of this data is used as train data and 15% as dev data
train_val_split = tokenized_dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = train_val_split["train"]
dev_dataset = train_val_split["test"]


# Create a DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'dev': dev_dataset,
})

dataset_dict

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 28348
    })
    dev: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5003
    })
})

In [7]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import numpy as np

# load model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Data Collator
# Padd alle Batches auf die längste Sequenz im Batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# Define metric function
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}



model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at deepset/gbert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
from huggingface_hub import notebook_login
notebook_login()

In [16]:
# not necessary
#!pip install torch --upgrade

In [9]:

# Define TrainingArguments
from transformers import TrainingArguments, Trainer
import os

training_args = TrainingArguments(
    output_dir="./results_flausch_classification_" + model_name,
    learning_rate=2e-5,
    per_device_train_batch_size=16, # Reduziere um OOM zu vermeiden
    per_device_eval_batch_size=16,  # Reduziere um OOM zu vermeiden
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    logging_steps=500, # Alle 500 Schritte loggen
    fp16=True, # Für schnellere und speichereffizientere Berechnung auf GPU
    gradient_checkpointing=True, # Hilft gegen OOM bei großen Modellen, macht Training langsamer
    save_total_limit=1, # Nur das beste Modell speichern
    report_to="none", # Wichtig: W&B und andere Logger deaktivieren, falls nicht gewünscht
    push_to_hub=False, # Zuerst trainieren, dann manuell pushen
    metric_for_best_model="eval_f1", # competition ranking metric
    greater_is_better=True, # Setze auf False, wenn du Loss minimieren willst
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)




<ipython-input-9-98d443c2fb30>:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
print("\nPredictions on dev set before training.")

predictions_output = trainer.predict(test_dataset=dataset_dict["dev"])
print(predictions_output.metrics)


Predictions on dev set before training.


{'test_loss': 0.7166978716850281, 'test_model_preparation_time': 0.0053, 'test_accuracy': 0.4655206875874475, 'test_f1': 0.47441442635855624, 'test_runtime': 19.1878, 'test_samples_per_second': 260.739, 'test_steps_per_second': 16.312}


In [11]:
# on colab with T4 ~60 minutes
trainer.train()

print("\nTraining abgeschlossen.")
print("Bestes Modell wurde am Ende des Trainings geladen.")


Step,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
500,0.284100,0.260128,0.005300,0.913652,0.914717


Step,Training Loss,Validation Loss,Model Preparation Time,Accuracy,F1
500,0.284100,0.260128,0.005300,0.913652,0.914717
1000,0.251400,0.222844,0.005300,0.921847,0.922672
1500,0.237100,0.203040,0.005300,0.935639,0.935247
2000,0.207900,0.280018,0.005300,0.926644,0.924666
2500,0.187800,0.228882,0.005300,0.937038,0.936550
3000,0.172600,0.224464,0.005300,0.937637,0.937203
3500,0.172700,0.182208,0.005300,0.938837,0.938812
4000,0.118200,0.278323,0.005300,0.938837,0.938301
4500,0.124100,0.241675,0.005300,0.941635,0.941430
5000,0.125400,0.228327,0.005300,0.941835,0.941722



Training abgeschlossen.
Bestes Modell wurde am Ende des Trainings geladen.


In [12]:

trainer.push_to_hub()

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Wiebke/results_flausch_classification_gbert-large_spelling_corrected/commit/c7f4266a255452638ad0999c9d720cf2b3f8e493', commit_message='End of training', commit_description='', oid='c7f4266a255452638ad0999c9d720cf2b3f8e493', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Wiebke/results_flausch_classification_gbert-large_spelling_corrected', endpoint='https://huggingface.co', repo_type='model', repo_id='Wiebke/results_flausch_classification_gbert-large_spelling_corrected'), pr_revision=None, pr_num=None)

In [13]:
print("\nPredictions on dev set after training.")

predictions_output = trainer.predict(test_dataset=dataset_dict["dev"])
print(predictions_output)


Predictions on dev set after training.


PredictionOutput(predictions=array([[ 3.2597656, -3.4023438],
       [ 2.7050781, -2.4492188],
       [ 3.2539062, -3.28125  ],
       ...,
       [ 3.5625   , -3.7695312],
       [ 3.4101562, -3.8339844],
       [ 3.0410156, -3.3007812]], dtype=float32), label_ids=array([0, 0, 0, ..., 0, 0, 0]), metrics={'test_loss': 0.22832679748535156, 'test_model_preparation_time': 0.0053, 'test_accuracy': 0.9418348990605636, 'test_f1': 0.941721888912358, 'test_runtime': 20.5902, 'test_samples_per_second': 242.98, 'test_steps_per_second': 15.201})


## Playing around with trained models for task 1 and task 2

In [ ]:
my_checkpoint = "Wiebke/results_flausch_classification"
from transformers import pipeline

pipe = pipeline("text-classification", model=my_checkpoint)

texte= ["der Hund hat den Fisch geklaut","Ich finde das hast du echt gut gemacht",
       "wir Fortunen sind die besten", "geil, auf so nen Murks kommst nur du"]

pipe(texte)

config.json:   0%|          | 0.00/750 [00:00<?, ?B/s]

c:\Users\Wiebke Petersen\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Wiebke Petersen\.cache\huggingface\hub\models--Wiebke--results_flausch_classification. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/726k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


[{'label': 'no', 'score': 0.9997480511665344},
 {'label': 'yes', 'score': 0.9985883831977844},
 {'label': 'yes', 'score': 0.9956735968589783},
 {'label': 'yes', 'score': 0.7363585829734802}]

In [ ]:
my_span_checkpoint = "Wiebke/flausch_span_model"
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe_token_class = pipeline("token-classification", model=my_span_checkpoint)

pipe_token_class(texte)

config.json:   0%|          | 0.00/1.73k [00:00<?, ?B/s]

c:\Users\Wiebke Petersen\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Wiebke Petersen\.cache\huggingface\hub\models--Wiebke--flausch_span_model. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/434M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/255k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/726k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cpu


[[],
 [{'entity': 'B-compliment',
   'score': 0.99295324,
   'index': 1,
   'word': 'Ich',
   'start': 0,
   'end': 3},
  {'entity': 'I-compliment',
   'score': 0.99763834,
   'index': 2,
   'word': 'finde',
   'start': 4,
   'end': 9},
  {'entity': 'I-compliment',
   'score': 0.997959,
   'index': 3,
   'word': 'das',
   'start': 10,
   'end': 13},
  {'entity': 'I-compliment',
   'score': 0.99804556,
   'index': 4,
   'word': 'hast',
   'start': 14,
   'end': 18},
  {'entity': 'I-compliment',
   'score': 0.99799514,
   'index': 5,
   'word': 'du',
   'start': 19,
   'end': 21},
  {'entity': 'I-compliment',
   'score': 0.9982108,
   'index': 6,
   'word': 'echt',
   'start': 22,
   'end': 26},
  {'entity': 'I-compliment',
   'score': 0.99797326,
   'index': 7,
   'word': 'gut',
   'start': 27,
   'end': 30},
  {'entity': 'I-compliment',
   'score': 0.997894,
   'index': 8,
   'word': 'gemacht',
   'start': 31,
   'end': 38}],
 [{'entity': 'B-affection declaration',
   'score': 0.958662

In [ ]:
texte= ["der Hund hat den Fisch geklaut","Ich finde das hast du echt gut gemacht",
       "wir Fortunen sind die besten", "geil, auf so nen Murks kommst nur du","da hast du vollen Bockmist gebaut",
       "das ist voll der Mist", "das wird der neue heiße Scheiß"]


In [ ]:
out = pipe(texte)
for i in range(len(texte)):
    print(out[i]["label"], " : " , texte[i])

no  :  der Hund hat den Fisch geklaut
yes  :  Ich finde das hast du echt gut gemacht
yes  :  wir Fortunen sind die besten
yes  :  geil, auf so nen Murks kommst nur du
no  :  da hast du vollen Bockmist gebaut
no  :  das ist voll der Mist
no  :  das wird der neue heiße Scheiß


In [ ]:
out = pipe_token_class(texte)
for i in range(len(texte)):
    print(texte[i])
    for j in out[i]:
        if j["score"] > 0.5: # nur wenn score > 0.5
            print(j["word"], " : ", j["entity"], " : ", j["score"])
    print("\n")

der Hund hat den Fisch geklaut


Ich finde das hast du echt gut gemacht
Ich  :  B-compliment  :  0.99295324
finde  :  I-compliment  :  0.99763834
das  :  I-compliment  :  0.997959
hast  :  I-compliment  :  0.99804556
du  :  I-compliment  :  0.99799514
echt  :  I-compliment  :  0.9982108
gut  :  I-compliment  :  0.99797326
gemacht  :  I-compliment  :  0.997894


wir Fortunen sind die besten
wir  :  B-affection declaration  :  0.9586628
Fort  :  I-affection declaration  :  0.9499238
##unen  :  I-affection declaration  :  0.97038966
sind  :  I-affection declaration  :  0.9810565
die  :  I-affection declaration  :  0.9812811
besten  :  I-affection declaration  :  0.9906984


geil, auf so nen Murks kommst nur du
ge  :  B-positive feedback  :  0.9935782
##il  :  I-positive feedback  :  0.99423605


da hast du vollen Bockmist gebaut


das ist voll der Mist
das  :  B-positive feedback  :  0.7710201
ist  :  I-positive feedback  :  0.5849939
der  :  I-positive feedback  :  0.5908547
##t  :  I-po